In [1]:
import pandas as pd 
import os
from google.auth.transport.requests import Request # Google api modules, used to call various services within network (sheets, gmail)
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from itertools import permutations # creating permutations of tasks on the chore list 
from random import randrange # used for order selection
import datetime as dt
import smtplib # used for email sending/construction
from email.message import EmailMessage
from email import errors
from dotenv import load_dotenv # env file loading 
from tasks import * # importing classes for the chores created in a separate script.
import base64 # encoding and decoding emails before sending 


In [ ]:
load_dotenv()

SPREADSHEET_ID = os.getenv("SPREADSHEET_ID")

SAMPLE_RANGE_NAME = os.getenv("SAMPLE_RANGE_NAME")

SCOPE = ["https://www.googleapis.com/auth/spreadsheets", 
            "https://www.googleapis.com/auth/gmail.addons.current.action.compose", 
            "https://www.googleapis.com/auth/gmail.compose"]

creds = None 


if os.path.exists("token.json"): 
    creds = Credentials.from_authorized_user_file("token.json", SCOPE) # if token file exists, use it and scope for credentials 
if not creds or not creds.valid: 
      if creds and creds.expired and creds.refresh_token: 
        creds.refresh(Request())
else: 
    flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPE)  
    creds = flow.run_local_server(port=0)
    
    with open("token.json", "w") as token: 
        token.write(creds.to_json())

In [2]:
def first_row_header(df) -> pd.DataFrame : 
    """ Turn first row of dataset into column names. Deletes unwanted first row after making them into the column names. """
    try:
        df.columns = df.iloc[0] 
        df = df[1:]
        return df
    except Exception as e: 
        return f"There has been an error: {e}"
    

def fetch_past_week(SPREADSHEET_ID, SAMPLE_RANGE_NAME, creds): 
    """ Gets the past week's worth of data, to be used for comparison in determining the next set of tasks for the list of contacts. """
    try: 

        service = build("sheets", "v4", credentials=creds) 
        sheet = service.spreadsheets()
        result = (sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=SAMPLE_RANGE_NAME).execute())
        values = result.get("values", [])

        if not values: 
            return("No data found.") 
        else: 
            values = first_row_header(pd.DataFrame(values))
            values = list(values.tail(1).iloc[0, 1:])
            values['rownum'] = values.shape[0] + 2
            return values
    except Exception as e: 
         return f"Error during initial request: {e}"
    

def next_task_set(most_recent, task_list = ["Kitchen", "Dining", "Landing"]): 
    """ Determines the order of tasks to be completed in the next week, using the past week's task as a reference. """    
    new_order = []
    combs_new = [list(x) for x in permutations(task_list)]

    for item in combs_new: 
        if combs_new == most_recent: 
            pass
        else: 
            new_order.append(item)

    new_index = randrange(0, len(new_order))

    tba =  new_order[new_index]
    return tba